# Mã hóa theo nhãn cho 13 và 24 cột

Bước 7–8 của kế hoạch S6E11.

So sánh GLM, LightGBM và XGBoost thêm TE13 với ba đối chứng GPU chỉ dùng đặc trưng gốc. TE ước lượng tỷ lệ gian lận có làm trơn bằng 20 mẫu. Năm fold nội bộ tách nhãn của từng dòng huấn luyện khỏi cách mã hóa dòng đó. GLM dùng logit của TE, cây dùng tỷ lệ. T04 mở rộng mô hình TE13 dẫn đầu sang 24 cột.

## Chạy lại

Chạy từ thư mục `plan_runs` với môi trường Python đã dùng cho RTX 4060. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```powershell
& '..\.build\venv\Scripts\python.exe' -B -X utf8 run_plan.py --stage te
```

Dùng `--stage te --only-te T01`, `T02` hoặc `T03` để chạy một cặp thí nghiệm và đối chứng. T04 cần điểm của cả T01–T03 để chọn họ mô hình.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'te']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `T04` | `T02` | 0.948574 | +0.000047 | 0.520055 | 0.044973 | 487.1 | 0.947379 | 0.950095 | 0.953644 | 0.949273 | 0.942742 |
| `T02` | `GPU-S6-L04` | 0.948527 | +0.001295 | 0.518542 | 0.045057 | 406.6 | 0.947342 | 0.949910 | 0.953597 | 0.949303 | 0.942775 |
| `T03` | `GPU-S6-X02` | 0.948495 | +0.001356 | 0.521020 | 0.044963 | 138.3 | 0.947440 | 0.949946 | 0.953561 | 0.948919 | 0.942902 |
| `T01` | `GPU-B02` | 0.947261 | +0.005371 | 0.513583 | 0.045406 | 19.4 | 0.946496 | 0.948613 | 0.952620 | 0.947446 | 0.941290 |
| `GPU-S6-L04` | — | 0.947232 | — | 0.513262 | 0.045409 | 591.6 | 0.945810 | 0.948485 | 0.951833 | 0.947570 | 0.942810 |
| `GPU-S6-X02` | — | 0.947139 | — | 0.513930 | 0.045408 | 148.9 | 0.945917 | 0.948347 | 0.951603 | 0.947423 | 0.942902 |
| `GPU-B02` | — | 0.941890 | — | 0.489712 | 0.047038 | 20.6 | 0.941376 | 0.943491 | 0.946932 | 0.941524 | 0.936274 |

## Đọc dữ liệu kết quả hiện tại

Cell này xác minh những lượt hoàn tất từ dự đoán OOF và tệp metric.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import final_summary, read_stage

data_dir = plan_dir / "results" / "full"
rows, pending, notes = read_stage('01-te', data_dir)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(data_dir, markdown_links=False)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    directory = data_dir / record["name"]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    with np.load(directory / "predictions.npz", allow_pickle=False) as saved:
        display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

T04 chọn họ mô hình theo chính điểm khảo sát này. Kết quả cần được đọc theo từng đối chứng.